# Setup


In [44]:
# Optional autoreload while developing locally

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [45]:
# to load the virtual environment
## & "$HOME\nlp_owlapy_env_py311\Scripts\Activate.ps1"

In [46]:

# 0) Local-only setup and base diagnostics
import os
import sys
import platform
from pathlib import Path

print("Python:", sys.version)
print("Platform:", platform.platform())

PROJECT_DIR = Path.cwd()



Python: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
Platform: Windows-10-10.0.26200-SP0


In [47]:

# 0.3) Shared device diagnostics
import torch

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU count:", torch.cuda.device_count())
    print("GPU 0:", torch.cuda.get_device_name(0))
    DEVICE = "cuda:0"
else:
    DEVICE = "cpu"
print("Using DEVICE =", DEVICE)


torch: 2.11.0+cu128
CUDA available: True
GPU count: 1
GPU 0: NVIDIA RTX A2000 8GB Laptop GPU
Using DEVICE = cuda:0


# Imports


In [48]:
import pickle
import re

import pandas as pd
from spacy.tokens import Doc

from dataclasses import dataclass, asdict
from itertools import product

from tokenizer import create_tokenizer, ensure_booknlp_extensions, tokens_to_dicts
from doc_chunker import create_chunker
from runtime_config import (
    RUNTIME_PROFILE,
    DEVICE,
    CHUNK_SIZE,
    OVERLAP_SENTENCES,
    MAX_EXPANDED_CHUNK_TOKENS,
)

In [49]:
from entity_extraction.entity_extraction_sub_orchestrator import (
    run_entity_extraction,
    print_non_singleton_entity_clusters,
)

from annotation_layer.spacy_extension import (
    register_spacy_annotation_extension,
    require_annotation_layer,
    require_entities,
    require_relations,
)


In [50]:
from ocean.ocean_probability_scoring import (
    ContextConfig,
    MentionRenderingConfig,
    OceanScoringConfig,
    OceanWeightConfig,
    DirectNLIConfig,
    OceanProbabilityExportConfig,
    export_ocean_probability_csvs,
)

from ocean.ocean_annotator import attach_ocean_from_folder


In [51]:
from cluster_typing.cluster_typing_probability_scoring import (
    ClusterTypingTraversalConfig,
    ClusterTypingScoringConfig,
    ClusterTypingMentionWeightConfig,
    ClusterTypingEvidenceExportConfig,
    export_cluster_typing_evidence_jsonls,
)

from cluster_typing.cluster_typing_annotator import (
    ClusterTypingAnnotationConfig,
    attach_cluster_typing_from_folder,
)

from cluster_typing.graph_contract import (
    class_human_readable_label,
    class_label,
    ontology_descendants,
    resolve_class_label,
)


In [52]:
from ontology.ontology_management import (
    assert_cluster_relation,
    assert_cluster_type,
    assert_cluster_value,
    build_relation_router,
    load_tbox,
    save_ontology,
)


In [53]:
from relationship_extraction.extract_relation_candidates import (
    extract_relation_candidates,
    export_routed_relation_candidates_jsonl,
)

from relationship_extraction.align_relation_assignments import (
    RelationNLIConfig,
    load_relation_nli_selector,
    export_relation_assignments_csv,
)

from relationship_extraction.aggregate_cluster_assertions import (
    RelationAggregationConfig,
    export_cluster_assertions_csv,
)

from relationship_extraction.annotate_relation_layer import (
    attach_relations_from_files,
)


# Functions


In [54]:
# text loading + preprocessing
def load_txt(path):
    path = Path(path)

    try:
        return path.read_text(encoding="utf-8")
    except UnicodeDecodeError:
        return path.read_text(encoding="latin-1")

In [55]:
import re

def preprocess_for_NER(text):
    # Normalize Windows/Mac newlines
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # Remove possible invisible BOM
    text = text.replace("\ufeff", "")

    # Normalize curly apostrophes/quotes
    text = text.replace("’", "'")
    text = text.replace("‘", "'")
    text = text.replace("“", '"').replace("”", '"')

    # Remove standalone page numbers, e.g.
    # This removes only lines that contain digits plus optional whitespace.
    text = re.sub(r"(?m)^[ \t]*\d+[ \t]*$", "", text)

    # Replace line breaks inside paragraphs with spaces,
    # but preserve paragraph boundaries
    paragraphs = text.split("\n\n")
    paragraphs = [
        re.sub(r"\s*\n\s*", " ", paragraph).strip()
        for paragraph in paragraphs
        if paragraph.strip()
    ]

    text = "\n\n".join(paragraphs)

    chapter_re = re.compile(
        r"(?im)^\s*chapter\s+(?:[ivxlcdm]+|\d+)\b[^\n]*\n*"
    )

    text = chapter_re.sub("", text)

    # Normalize multiple spaces
    text = re.sub(r"[ \t]+", " ", text)

    return text.strip()

In [56]:

def resolve_text_path(
    *,
    project_dir: Path,
    filename: str,
    local_path: str | Path,
) -> Path:
    """Resolve the local input text path."""
    text_path = Path(local_path)
    if text_path.exists():
        return text_path

    fallback_path = project_dir / filename
    if fallback_path.exists():
        return fallback_path

    raise FileNotFoundError(
        "Could not resolve a local text file. "
        f"Tried LOCAL_TEXT_PATH={text_path} and PROJECT_DIR / {filename!r} = {fallback_path}."
    )


In [57]:
def save_doc(doc, path: str | Path) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("wb") as f:
        pickle.dump(doc, f)
    return path


def load_doc(path: str | Path):
    # BookNLP/tokenizer extensions and the unified annotation_layer extension
    # must be registered before unpickling the Doc.
    ensure_booknlp_extensions()
    register_spacy_annotation_extension()
    
    path = Path(path)
    with path.open("rb") as f:
        return pickle.load(f)


# Config


## I/O config


In [ ]:
# Requested local file
TEXT_FILENAME = "oz_full.txt"
LOCAL_TEXT_PATH = Path(f"./resources/books/{TEXT_FILENAME}")

TEXT_PATH = resolve_text_path(
    project_dir=PROJECT_DIR,
    filename=TEXT_FILENAME,
    local_path=LOCAL_TEXT_PATH,
)

OUTPUT_ROOT = PROJECT_DIR / f"outputs_[{TEXT_FILENAME}]"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
ONTOLOGY_ROOT = PROJECT_DIR / "resources/ontologies"

TOKENIZED_DOC_PATH = OUTPUT_ROOT / "tokenized_doc.pkl"
ENTITY_ANNOTATED_DOC_PATH = OUTPUT_ROOT / "entity_annotated_doc.pkl"
TYPED_ENTITY_DOC_PATH = OUTPUT_ROOT / "typed_entity_doc.pkl"
OCEAN_ENTITY_DOC_PATH = OUTPUT_ROOT / "ocean_entity_doc.pkl"
RELATION_ANNOTATED_DOC_PATH = OUTPUT_ROOT / "relation_annotated_doc.pkl"

ONTOLOGY_TTL_PATH = ONTOLOGY_ROOT / "narrative_entity_ontology_clean.ttl"

RELATION_OUTPUT_DIR = OUTPUT_ROOT / "relations"
RELATION_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ROUTED_RELATION_CANDIDATES_PATH = RELATION_OUTPUT_DIR / "routed_relation_candidates.jsonl"
RELATION_ASSIGNMENTS_PATH = RELATION_OUTPUT_DIR / "relation_assignments.csv"
CLUSTER_ASSERTIONS_PATH = RELATION_OUTPUT_DIR / "cluster_assertions.csv"
POPULATED_ONTOLOGY_PATH = OUTPUT_ROOT / "populated_ontology.ttl"

print("TEXT_PATH =", TEXT_PATH)
print("OUTPUT_ROOT =", OUTPUT_ROOT)
print("TOKENIZED_DOC_PATH =", TOKENIZED_DOC_PATH)
print("ENTITY_ANNOTATED_DOC_PATH =", ENTITY_ANNOTATED_DOC_PATH)
print("TYPED_ENTITY_DOC_PATH =", TYPED_ENTITY_DOC_PATH)
print("OCEAN_ENTITY_DOC_PATH =", OCEAN_ENTITY_DOC_PATH)
print("RELATION_ANNOTATED_DOC_PATH =", RELATION_ANNOTATED_DOC_PATH)
print("ONTOLOGY_TTL_PATH =", ONTOLOGY_TTL_PATH)


TEXT_PATH = resources\books\cosmicomics_a_sign_in_space.txt
OUTPUT_ROOT = c:\Users\Bobby\Desktop\real_shit\NLP_character_graph_pipeline_annotation_layer_refactor\outputs_[cosmicomics_a_sign_in_space.txt]
TOKENIZED_DOC_PATH = c:\Users\Bobby\Desktop\real_shit\NLP_character_graph_pipeline_annotation_layer_refactor\outputs_[cosmicomics_a_sign_in_space.txt]\tokenized_doc.pkl
ENTITY_ANNOTATED_DOC_PATH = c:\Users\Bobby\Desktop\real_shit\NLP_character_graph_pipeline_annotation_layer_refactor\outputs_[cosmicomics_a_sign_in_space.txt]\entity_annotated_doc.pkl
TYPED_ENTITY_DOC_PATH = c:\Users\Bobby\Desktop\real_shit\NLP_character_graph_pipeline_annotation_layer_refactor\outputs_[cosmicomics_a_sign_in_space.txt]\typed_entity_doc.pkl
OCEAN_ENTITY_DOC_PATH = c:\Users\Bobby\Desktop\real_shit\NLP_character_graph_pipeline_annotation_layer_refactor\outputs_[cosmicomics_a_sign_in_space.txt]\ocean_entity_doc.pkl
RELATION_ANNOTATED_DOC_PATH = c:\Users\Bobby\Desktop\real_shit\NLP_character_graph_pipeline_an

In [59]:
GLOBAL_ENTITY_OUTPUT_DIR = OUTPUT_ROOT / "global_entities"
GLOBAL_ENTITY_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("GLOBAL_ENTITY_OUTPUT_DIR =", GLOBAL_ENTITY_OUTPUT_DIR)


GLOBAL_ENTITY_OUTPUT_DIR = c:\Users\Bobby\Desktop\real_shit\NLP_character_graph_pipeline_annotation_layer_refactor\outputs_[cosmicomics_a_sign_in_space.txt]\global_entities


## Runtime and pipeline config


In [60]:
# Pipeline configuration
SPACY_MODEL = "en_core_web_sm"
TOKENIZER_DISABLE = ("ner",)

OCEAN_ROOT_CLASS_LABEL = "Character"

RELATION_PAIR_BATCH_SIZE = 64
RELATION_OVERWRITE_STAGE_1 = False
RELATION_OVERWRITE_STAGE_2 = False
RELATION_RESUME_STAGE_2 = True
RELATION_OVERWRITE_STAGE_3 = True

print("RUNTIME_PROFILE =", RUNTIME_PROFILE)
print("DEVICE =", DEVICE)
print("CHUNK_SIZE(expanded) =", CHUNK_SIZE)
print("OVERLAP_SENTENCES =", OVERLAP_SENTENCES)
print("MAX_EXPANDED_CHUNK_TOKENS =", MAX_EXPANDED_CHUNK_TOKENS)
print("GLOBAL_ENTITY_OUTPUT_DIR =", GLOBAL_ENTITY_OUTPUT_DIR)


RUNTIME_PROFILE = {'kind': 'local_cuda', 'env': 'Local', 'cuda_available': True, 'gpu_count': 1, 'gpu_name': 'NVIDIA RTX A2000 8GB Laptop GPU', 'device': 'cuda:0', 'cpu_load_first': True, 'precision_policy': 'auto', 'p100_fallback_to_float32': False, 'default_chunk_size': 6000, 'default_overlap_sentences': 60}
DEVICE = cuda:0
CHUNK_SIZE(expanded) = 6000
OVERLAP_SENTENCES = 60
MAX_EXPANDED_CHUNK_TOKENS = 6000
GLOBAL_ENTITY_OUTPUT_DIR = c:\Users\Bobby\Desktop\real_shit\NLP_character_graph_pipeline_annotation_layer_refactor\outputs_[cosmicomics_a_sign_in_space.txt]\global_entities


# Main


### Tokenization


In [61]:
# Load or create the stable tokenized Doc artifact.
if TOKENIZED_DOC_PATH.exists():
    print(f"[doc] Loading tokenized Doc from {TOKENIZED_DOC_PATH}")
    doc = load_doc(TOKENIZED_DOC_PATH)
else:
    print("[doc] Tokenized Doc not found; creating it from raw text.")
    raw_text = load_txt(TEXT_PATH)
    text = preprocess_for_NER(raw_text)
    tokenizer = create_tokenizer(
        SPACY_MODEL,
        disable=TOKENIZER_DISABLE,
        verbose=True,
    )
    doc = tokenizer.tokenize(text)
    save_doc(doc, TOKENIZED_DOC_PATH)
    print(f"[doc] Saved tokenized Doc to {TOKENIZED_DOC_PATH}")

print("Doc tokens:", len(doc))
print("Doc sentences:", sum(1 for _ in doc.sents))

[doc] Loading tokenized Doc from c:\Users\Bobby\Desktop\real_shit\NLP_character_graph_pipeline_annotation_layer_refactor\outputs_[cosmicomics_a_sign_in_space.txt]\tokenized_doc.pkl
Doc tokens: 4092
Doc sentences: 98


### Chunking


In [62]:
chunker = create_chunker(
    chunk_size=CHUNK_SIZE,
    overlap_sentences=OVERLAP_SENTENCES,
    max_expanded_chunk_tokens=MAX_EXPANDED_CHUNK_TOKENS,
)
chunk_plan = chunker.plan(doc)

### Ontology building


In [63]:
onto, class_graph = load_tbox(ONTOLOGY_TTL_PATH, require_property_descriptions=True)

print(class_graph.number_of_nodes(), "classes")
print(class_graph.number_of_edges(), "subclass edges")

relation_router = build_relation_router(
    onto=onto,
    ontology_path=ONTOLOGY_TTL_PATH,
    class_graph=class_graph,
)

print("Object properties:", len(relation_router.relation_specs))


23 classes
22 subclass edges
Object properties: 16


## Node extraction


### Entity clusters extraction


In [64]:
if ENTITY_ANNOTATED_DOC_PATH.exists():
    print(f"[entity-extraction] Loading entity-annotated Doc from {ENTITY_ANNOTATED_DOC_PATH}")
    doc = load_doc(ENTITY_ANNOTATED_DOC_PATH)
else:
    print("[entity-extraction] Entity-annotated Doc not found; calling entity extraction sub-orchestrator.")
    doc = run_entity_extraction(
        doc=doc,
        chunker=chunker,
        chunk_plan=chunk_plan,
        document_id=TEXT_PATH.stem,
        output_dir=GLOBAL_ENTITY_OUTPUT_DIR,
    )
    save_doc(doc, ENTITY_ANNOTATED_DOC_PATH)
    print("[entity-extraction] Saved annotated Doc to:", ENTITY_ANNOTATED_DOC_PATH)


[entity-extraction] Loading entity-annotated Doc from c:\Users\Bobby\Desktop\real_shit\NLP_character_graph_pipeline_annotation_layer_refactor\outputs_[cosmicomics_a_sign_in_space.txt]\entity_annotated_doc.pkl


In [65]:
ann = require_annotation_layer(doc)
entities = ann.require_entities()

print("[entity-extraction] Summary:")
print(ann.summary())
print_non_singleton_entity_clusters(doc)


[entity-extraction] Summary:
{'has_entities': True, 'has_cluster_typing': False, 'has_ocean': False, 'has_relations': False, 'n_mentions': 251, 'n_clusters': 3, 'n_typed_clusters': 0, 'n_ocean_clusters': 0, 'n_indexed_tokens': 264, 'n_indexed_heads': 251, 'n_indexed_spans': 251}
Non-singleton clusters: 3

cluster_id=1 | canonical_name='I' | n_mentions=210

cluster_id=0 | canonical_name='space' | n_mentions=21

cluster_id=2 | canonical_name='Kgwgk' | n_mentions=20



### Ontology cluster typing


In [66]:
ontology_n_mentions = None  # None = type ALL mentions for each requested cluster
ontology_random_seed = 67

In [67]:
if TYPED_ENTITY_DOC_PATH.exists():
    print(f"[entity typing] Loading typed-entity Doc from {TYPED_ENTITY_DOC_PATH}")
    doc = load_doc(TYPED_ENTITY_DOC_PATH)
else:
    if not ONTOLOGY_TTL_PATH.exists():
        raise FileNotFoundError(
            f"Ontology TTL not found: {ONTOLOGY_TTL_PATH}. "
            "Create a networkx.DiGraph by any external method, or place the TTL here."
        )

    jsonl_paths_by_cluster_id = export_cluster_typing_evidence_jsonls(
        doc,
        class_graph,
        ClusterTypingEvidenceExportConfig(
            n_mentions_per_cluster=ontology_n_mentions,
            output_root=OUTPUT_ROOT,

            context_config=ContextConfig(
                n_sentences_before=0,
                n_sentences_after=0,
                mark_mention=True,
                deduplicate=False,
            ),
            rendering_config=MentionRenderingConfig(
                canonicalize_simple_mentions=True,
                keep_first_second_person=True,
            ),
            traversal_config=ClusterTypingTraversalConfig(
                skip_single_root=True,
                include_stay_option=True,
                force_leaf=False,
            ),
            scoring_config=ClusterTypingScoringConfig(
                subject_aware=True,
            ),
            mention_weight_config=ClusterTypingMentionWeightConfig(),
            nli_config=DirectNLIConfig(
                pair_batch_size=64,
            ),

            chunk_size=16,

            overwrite_jsonl=True,
            resume_from_jsonl=False,

            random_seed=ontology_random_seed,
            sort_sample_by_cluster_order=True,
            print_progress=True,
        ),
    )

    print(jsonl_paths_by_cluster_id)

    n_part = "all" if ontology_n_mentions is None else str(ontology_n_mentions)
    ontology_folder = OUTPUT_ROOT / "cluster_typing" / n_part

    attach_cluster_typing_from_folder(
        doc,
        class_graph,
        ontology_folder,
        config=ClusterTypingAnnotationConfig(
            use_mention_weight=True,
        ),
        overwrite=True,
    )
    save_doc(doc, TYPED_ENTITY_DOC_PATH)
    print("[entity typing] Saved annotated Doc to:", TYPED_ENTITY_DOC_PATH)

ann = require_annotation_layer(doc)
entities = ann.require_entities()
print(ann.summary())


[entity typing] Loading typed-entity Doc from c:\Users\Bobby\Desktop\real_shit\NLP_character_graph_pipeline_annotation_layer_refactor\outputs_[cosmicomics_a_sign_in_space.txt]\typed_entity_doc.pkl
{'has_entities': True, 'has_cluster_typing': True, 'has_ocean': False, 'has_relations': False, 'n_mentions': 251, 'n_clusters': 3, 'n_typed_clusters': 3, 'n_ocean_clusters': 0, 'n_indexed_tokens': 264, 'n_indexed_heads': 251, 'n_indexed_spans': 251}


In [68]:
ann = require_annotation_layer(doc)
entities = ann.require_entities()

rows = []

for cluster_id, cluster in sorted(entities.clusters.items()):
    typing = cluster.typing
    class_iri = typing.class_iri if typing is not None else None
    rows.append(
        {
            "cluster_id": cluster_id,
            "canonical_name": cluster.canonical_name,
            "n_mentions": len(cluster.mention_ids),
            "class_iri": class_iri,
            "ontology_class_label": class_label(class_graph, class_iri) if class_iri else None,
            "ontology_class_human_readable_label": class_human_readable_label(class_graph, class_iri) if class_iri else None,
        }
    )

ontology_clusters_df = pd.DataFrame(rows)

ontology_clusters_df


,cluster_id,canonical_name,n_mentions,class_iri,ontology_class_label,ontology_class_human_readable_label
0,0,space,21,https://example.org/nlp-sdai/narrative-entity-...,Belief or value,Belief Or Value
1,1,I,210,https://example.org/nlp-sdai/narrative-entity-...,Human character,Human Character
2,2,Kgwgk,20,https://example.org/nlp-sdai/narrative-entity-...,Non-human character,Non Human Character


### OCEAN scoring


In [69]:
def entity_cluster_ids_of_class_or_subclass(entities, class_graph, class_label_ref: str):
    valid_class_iris = ontology_descendants(
        class_graph,
        class_label_ref,
        include_self=True,
    )

    return [
        cluster_id
        for cluster_id, cluster in entities.clusters.items()
        if cluster.typing is not None and cluster.typing.class_iri in valid_class_iris
    ]


In [70]:
ann = require_annotation_layer(doc)
entities = ann.require_entities()

cluster_ids = entity_cluster_ids_of_class_or_subclass(
    entities,
    class_graph,
    OCEAN_ROOT_CLASS_LABEL,
)

for cluster_id in sorted(cluster_ids):
    print(cluster_id, end=', ')


1, 2, 

In [71]:
n_mentions = None  # None = score ALL mentions for each requested cluster
random_seed = 67

In [72]:
if OCEAN_ENTITY_DOC_PATH.exists():
    print(f"[OCEAN scoring] Loading OCEAN-entity Doc from {OCEAN_ENTITY_DOC_PATH}")
    doc = load_doc(OCEAN_ENTITY_DOC_PATH)
else:
    csv_paths_by_cluster_id = export_ocean_probability_csvs(
        doc,
        OceanProbabilityExportConfig(
            cluster_ids=cluster_ids,
            n_mentions_per_cluster=n_mentions,
            output_root=OUTPUT_ROOT,

            context_config=ContextConfig(
                n_sentences_before=0,
                n_sentences_after=0,
                mark_mention=True,
                deduplicate=False,
            ),
            rendering_config=MentionRenderingConfig(
                canonicalize_simple_mentions=True,
                keep_first_second_person=True,
            ),
            scoring_config=OceanScoringConfig(
                subject_aware=True,
            ),
            weight_config=OceanWeightConfig(),
            nli_config=DirectNLIConfig(
                pair_batch_size=64, # on stronger hardware can be safely set to 64
            ),

            chunk_size=64,

            overwrite_csv=False,
            resume_from_csv=True,
            use_sqlite_cache=True,

            random_seed=random_seed,
            sort_sample_by_cluster_order=True,
            return_dataframes=False,
            print_progress=True,
        ),
    )

    print(csv_paths_by_cluster_id)

    n_part = "all" if n_mentions is None else str(n_mentions)
    ocean_folder = OUTPUT_ROOT / "OCEAN_profiles" / n_part

    attach_ocean_from_folder(
        doc,
        ocean_folder,
        overwrite=True,
    )
    save_doc(doc, OCEAN_ENTITY_DOC_PATH)
    print("[OCEAN scoring] Saved annotated Doc to:", OCEAN_ENTITY_DOC_PATH)

ann = require_annotation_layer(doc)
print(ann.summary())


[OCEAN scoring] Loading OCEAN-entity Doc from c:\Users\Bobby\Desktop\real_shit\NLP_character_graph_pipeline_annotation_layer_refactor\outputs_[cosmicomics_a_sign_in_space.txt]\ocean_entity_doc.pkl
{'has_entities': True, 'has_cluster_typing': True, 'has_ocean': True, 'has_relations': False, 'n_mentions': 251, 'n_clusters': 3, 'n_typed_clusters': 3, 'n_ocean_clusters': 2, 'n_indexed_tokens': 264, 'n_indexed_heads': 251, 'n_indexed_spans': 251}


In [73]:
ann = require_annotation_layer(doc)
entities = ann.require_entities()

for cluster_id in cluster_ids:
    cluster = entities.cluster(cluster_id)
    print(cluster.canonical_name, sep="->")
    print(cluster.ocean.scores.as_dict() if cluster.ocean is not None else None)


I
{'openness': 46.12, 'conscientiousness': 31.17, 'extraversion': 37.83, 'agreeableness': 44.79, 'neuroticism': 57.17}
Kgwgk
{'openness': 50.59, 'conscientiousness': 54.14, 'extraversion': 49.38, 'agreeableness': 19.16, 'neuroticism': 41.16}


## Edge extraction


In [74]:
for relation_iri in sorted(relation_router.relation_specs):
    print(relation_iri)


https://example.org/nlp-sdai/narrative-entity-ontology#accompanies
https://example.org/nlp-sdai/narrative-entity-ontology#containedInPlace
https://example.org/nlp-sdai/narrative-entity-ontology#containsPlace
https://example.org/nlp-sdai/narrative-entity-ontology#guides
https://example.org/nlp-sdai/narrative-entity-ontology#harms
https://example.org/nlp-sdai/narrative-entity-ontology#hasMember
https://example.org/nlp-sdai/narrative-entity-ontology#hasParticipant
https://example.org/nlp-sdai/narrative-entity-ontology#holdsAttitudeToward
https://example.org/nlp-sdai/narrative-entity-ontology#hostsEvent
https://example.org/nlp-sdai/narrative-entity-ontology#locatedIn
https://example.org/nlp-sdai/narrative-entity-ontology#memberOf
https://example.org/nlp-sdai/narrative-entity-ontology#movesToward
https://example.org/nlp-sdai/narrative-entity-ontology#objectUsedBy
https://example.org/nlp-sdai/narrative-entity-ontology#occursAt
https://example.org/nlp-sdai/narrative-entity-ontology#participat

In [75]:
def print_tree(g, root=None, indent=""):
    def label(n):
        data = g.nodes[n]
        return data.get("human_readable_label") or data.get("label") or str(n).split("#")[-1]

    if root is None:
        roots = [n for n in g.nodes if g.in_degree(n) == 0]
        for r in roots:
            print_tree(g, r)
        return

    print(indent + label(root))

    children = sorted(g.successors(root), key=label)
    for child in children:
        print_tree(g, child, indent + "    ")

print_tree(class_graph)

Narrative Entity
    Character
        Human Character
        Non Human Character
    Concept
        Belief Or Value
        Power Or Ability
        Rule Or Law
    Event
        Action
        Occurrence
    Group
        Informal Group
        Organization
        People Or Species
    Object
        Document
        Item
        Substance
    Place
        Region
        Settlement
        Site


In [76]:
# Inspect relation_router

import pandas as pd

def short(iri):
    iri = str(iri)
    return iri.rsplit("#", 1)[-1] if "#" in iri else iri.rstrip("/").rsplit("/", 1)[-1]

relation_specs = relation_router.relation_specs

router_df = pd.DataFrame([
    {
        "property": short(spec.iri),
        "domains": ", ".join(short(x) for x in spec.domains),
        "ranges": ", ".join(short(x) for x in spec.ranges),
        "label": spec.human_readable_label,
        "description": spec.description,
    }
    for spec in relation_specs.values()
])

print(f"Object properties in router: {len(router_df)}")
display(router_df.sort_values("property"))


Object properties in router: 16


,property,domains,ranges,label,description
15,accompanies,"Thing, Character, NarrativeEntity","Thing, Character, NarrativeEntity",Accompanies,Connects a character to another character with...
14,containedInPlace,"Thing, NarrativeEntity, Place","Thing, NarrativeEntity, Place",Contained In Place,Connects a place to a larger place that contai...
13,containsPlace,"Thing, NarrativeEntity, Place","Thing, NarrativeEntity, Place",Contains Place,Connects a place to a smaller place contained ...
12,guides,"Thing, Character, NarrativeEntity","Thing, NarrativeEntity",Guides,Connects a character to an entity that the cha...
11,harms,"Thing, Character, NarrativeEntity","Thing, NarrativeEntity",Harms,Connects a character to an entity that the cha...
10,hasMember,"Thing, Group, NarrativeEntity","Thing, NarrativeEntity",Has Member,Connects a group to one of its members.
9,hasParticipant,"Thing, Event, NarrativeEntity","Thing, NarrativeEntity",Has Participant,Connects an event to a narrative entity that p...
8,holdsAttitudeToward,"Thing, Character, NarrativeEntity","Thing, NarrativeEntity",Holds Attitude Toward,Connects a character to an entity toward which...
7,hostsEvent,"Thing, NarrativeEntity, Place","Thing, Event, NarrativeEntity",Hosts Event,Connects a place to an event that occurs there.
6,locatedIn,"Thing, NarrativeEntity","Thing, NarrativeEntity, Place",Located In,Connects a narrative entity to a place where i...


In [77]:
relation_candidates_df = extract_relation_candidates(doc)

relation_candidates_df[
    [
        "source_cluster_id",
        "source_canonical_name",
        "source_class_iri",
        "predicate",
        "target_cluster_id",
        "target_canonical_name",
        "target_class_iri",
        "sentence_text",
        "is_passive",
        "is_negated",
    ]
]

export_routed_relation_candidates_jsonl(
    doc=doc,
    relation_router=relation_router,
    output_path=ROUTED_RELATION_CANDIDATES_PATH,
    print_discards=True,
    overwrite=True,
)


[relation extraction] Wrote 4 routed candidates to c:\Users\Bobby\Desktop\real_shit\NLP_character_graph_pipeline_annotation_layer_refactor\outputs_[cosmicomics_a_sign_in_space.txt]\relations\routed_relation_candidates.jsonl


WindowsPath('c:/Users/Bobby/Desktop/real_shit/NLP_character_graph_pipeline_annotation_layer_refactor/outputs_[cosmicomics_a_sign_in_space.txt]/relations/routed_relation_candidates.jsonl')

In [78]:
selector = load_relation_nli_selector(
    RelationNLIConfig(
        pair_batch_size=RELATION_PAIR_BATCH_SIZE,
    )
)

export_relation_assignments_csv(
    input_path=ROUTED_RELATION_CANDIDATES_PATH,
    output_path=RELATION_ASSIGNMENTS_PATH,
    selector=selector,
    overwrite=RELATION_OVERWRITE_STAGE_2,
    resume=RELATION_RESUME_STAGE_2,
)

Loading weights: 100%|██████████| 394/394 [00:00<00:00, 8549.80it/s]


[relation alignment] Wrote 0 assignments to c:\Users\Bobby\Desktop\real_shit\NLP_character_graph_pipeline_annotation_layer_refactor\outputs_[cosmicomics_a_sign_in_space.txt]\relations\relation_assignments.csv (4 skipped by resume).


WindowsPath('c:/Users/Bobby/Desktop/real_shit/NLP_character_graph_pipeline_annotation_layer_refactor/outputs_[cosmicomics_a_sign_in_space.txt]/relations/relation_assignments.csv')

In [79]:
export_cluster_assertions_csv(
    assignments_path=RELATION_ASSIGNMENTS_PATH,
    output_path=CLUSTER_ASSERTIONS_PATH,
    aggregation_config=RelationAggregationConfig(
        aggregation_method="sum_softmax_by_cluster_pair",
        min_support_count=1,
        min_score=0.0,
    ),
    overwrite=RELATION_OVERWRITE_STAGE_3,
)

[relation aggregation] Wrote 2 cluster assertions to c:\Users\Bobby\Desktop\real_shit\NLP_character_graph_pipeline_annotation_layer_refactor\outputs_[cosmicomics_a_sign_in_space.txt]\relations\cluster_assertions.csv


WindowsPath('c:/Users/Bobby/Desktop/real_shit/NLP_character_graph_pipeline_annotation_layer_refactor/outputs_[cosmicomics_a_sign_in_space.txt]/relations/cluster_assertions.csv')

In [80]:
relation_layer = attach_relations_from_files(
    doc=doc,
    assignments_path=RELATION_ASSIGNMENTS_PATH,
    cluster_assertions_path=CLUSTER_ASSERTIONS_PATH,
    overwrite=True,
)

print(relation_layer.summary())


{'n_relation_instances': 4, 'n_relation_assignments': 4, 'n_relation_assertions': 2, 'n_source_cluster_indexes': 2, 'n_target_cluster_indexes': 2, 'n_property_indexes': 1}


In [81]:
ann = require_annotation_layer(doc)
entities = ann.require_entities()
relations = ann.require_relations()

for relation_id, assignment in relations.assignments.items():
    instance = relations.instance(relation_id)
    source_mention = entities.mention(instance.source_mention_id)
    target_mention = entities.mention(instance.target_mention_id)
    source = entities.cluster(source_mention.cluster_id)
    target = entities.cluster(target_mention.cluster_id)
    predicate = doc[instance.predicate_start:instance.predicate_end]
    prop = assignment.chosen_object_property_iri()
    conf = assignment.confidence()

    print(
        f"{source.canonical_name} -- {predicate.text} / {prop} "
        f"({conf:.3f}) --> {target.canonical_name}"
    )


I -- left that sign in / https://example.org/nlp-sdai/narrative-entity-ontology#holdsAttitudeToward (0.359) --> space
Kgwgk -- preceding / https://example.org/nlp-sdai/narrative-entity-ontology#movesToward (0.278) --> I
Kgwgk -- mock / https://example.org/nlp-sdai/narrative-entity-ontology#holdsAttitudeToward (0.457) --> I
I -- scattered these false signs liberally through / https://example.org/nlp-sdai/narrative-entity-ontology#holdsAttitudeToward (0.346) --> space


In [82]:
save_doc(doc, RELATION_ANNOTATED_DOC_PATH)


WindowsPath('c:/Users/Bobby/Desktop/real_shit/NLP_character_graph_pipeline_annotation_layer_refactor/outputs_[cosmicomics_a_sign_in_space.txt]/relation_annotated_doc.pkl')

In [83]:
def _local_name(iri: str) -> str:
    """Compact IRI rendering: http://x/y#livesIn -> livesIn."""
    return str(iri).rstrip("/#").rsplit("#", 1)[-1].rsplit("/", 1)[-1]


ann = require_annotation_layer(doc)
entities = ann.require_entities()
relations = ann.require_relations()

relations.rebuild_indexes()

assertions = sorted(
    relations.assertions.items(),
    key=lambda item: (
        int(item[1].source_cluster_id),
        _local_name(item[1].object_property_iri),
        int(item[1].target_cluster_id),
    ),
)

print("=" * 100)
print(f"RELATIONSHIP CLUSTERS: {len(assertions)}")
print("=" * 100)

for i, (assertion_id, assertion) in enumerate(assertions, start=1):
    source_id = int(assertion.source_cluster_id)
    target_id = int(assertion.target_cluster_id)

    source = entities.cluster(source_id)
    target = entities.cluster(target_id)
    prop_name = _local_name(assertion.object_property_iri)

    source_type = (
        class_human_readable_label(class_graph, source.typing.class_iri)
        if source.typing is not None else "UNKNOWN"
    )
    target_type = (
        class_human_readable_label(class_graph, target.typing.class_iri)
        if target.typing is not None else "UNKNOWN"
    )

    support_count = len(assertion.support_relation_ids)
    mean_conf = assertion.confidence

    print(f"{source.canonical_name}({source_type}) --[{prop_name}]--> {target.canonical_name}({target_type})")
    print(f"support_count={support_count} | confidence={mean_conf}")


RELATIONSHIP CLUSTERS: 2
I(Human Character) --[holdsAttitudeToward]--> space(Belief Or Value)
support_count=2 | confidence=0.7053988993790572
Kgwgk(Non Human Character) --[holdsAttitudeToward]--> I(Human Character)
support_count=2 | confidence=0.7146770572726391
